# Fase 5 · Wilcoxon: XGBoost vs LightGBM

**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la UJI**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Email** | mjmorteruiz@uoc.edu (UOC) / morte@uji.es (UJI) |
| **Fase** | Fase 5 — Modelado |
| **Tipo** | Análisis estadístico complementario |

---

## 🎯 Qué hace

Aplica el **test de Wilcoxon signed-rank pareado** entre XGBoost y LightGBM
(ambos con estrategia `none`, las dos primeras posiciones del ranking F5).

**Pregunta a responder:** ¿La diferencia entre XGBoost (1º) y LightGBM (4º) es
estadísticamente significativa, o es un empate técnico dentro de la incertidumbre
del CV?

## 📋 Requisitos

- `data/05_modelado/X_train.parquet`, `y_train.parquet` (m01a)
- `data/05_modelado/models/XGBoost__none.pkl` (m03)
- `data/05_modelado/models/LightGBM__none.pkl` (m03)

## 📤 Genera

| Archivo | Contenido |
|---|---|
| `data/05_modelado/results/wilcoxon_xgb_vs_lgb.json` | Resultado del test + folds |

## 🔄 Flujo

```
X_train, y_train (raw)
    ↓ 5-Fold StratifiedKFold (random_state=42, mismos folds para ambos)
    ↓ Por cada fold:
        ├─ XGBoost.fit(train_fold)  → predict_proba(val_fold) → AUC, F1
        └─ LightGBM.fit(train_fold) → predict_proba(val_fold) → AUC, F1
    ↓ Wilcoxon signed-rank pareado (5 valores AUC vs 5 valores AUC)
    ↓ Wilcoxon signed-rank pareado (5 valores F1 vs 5 valores F1)
→ wilcoxon_xgb_vs_lgb.json
```

## ⚠️ Decisiones metodológicas

1. **CV pareado** — mismos folds para ambos modelos (eliminación de varianza por split)
2. **`random_state=42`** — reproducibilidad total
3. **Datos originales** — los Pipeline `.pkl` incluyen su preprocesador interno
4. **`alternative='two-sided'`** — no asumimos previamente cuál gana
5. **α = 0.05** — nivel de significación estándar

## 📚 Interpretación del p-value

- `p < 0.05` → la diferencia ES significativa → aceptar el modelo con mejor métrica
- `p ≥ 0.05` → empate técnico → elegir por OTRO criterio (estabilidad, coherencia, eficiencia)

## ➡️ Siguiente

Documentar el resultado en la memoria del TFM (Sección 4 — Resultados, Anexo D).


In [1]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN Y CARGA DE DATOS
# ============================================================================

import sys, json, warnings
from pathlib import Path
import joblib
import numpy as np
import pandas as pd

# ── ROOT robusto ──────────────────────────────────────────────────────────────
ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'src').exists():
        break
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

# ── Imports del proyecto ──────────────────────────────────────────────────────
from src.utils import formato_numero_es
fmt = formato_numero_es

# ── Rutas ─────────────────────────────────────────────────────────────────────
RUTA_MODELADO = ROOT / 'data' / '05_modelado'
RUTA_MODELS   = RUTA_MODELADO / 'models'
RUTA_RESULTS  = RUTA_MODELADO / 'results'

# ── Cargar datos originales (los Pipeline incluyen su preprocesador) ─────────────
X_train = pd.read_parquet(RUTA_MODELADO / 'X_train.parquet')
y_train = pd.read_parquet(RUTA_MODELADO / 'y_train.parquet').squeeze()

print('═' * 60)
print('Datos cargados')
print('═' * 60)
print(f'X_train: {X_train.shape}')
print(f'y_train: {y_train.shape} | abandono: {(y_train==1).mean()*100:.1f}%')
print(f'\nROOT: {ROOT}')


════════════════════════════════════════════════════════════
Datos cargados
════════════════════════════════════════════════════════════
X_train: (26896, 27)
y_train: (26896,) | abandono: 29.2%

ROOT: c:\PRUEBAS\AU_UJI_v2_RUTA_B


In [2]:
# ============================================================================
# CELDA 2: CARGAR MODELOS XGBOOST Y LIGHTGBM (estrategia 'none')
# ============================================================================
# Cargamos los Pipeline ya entrenados como REFERENCIA de configuración.
# En el siguiente paso los clonaremos y reentrenaremos en cada fold del CV
# para garantizar que el test de Wilcoxon sea pareado y reproducible.
# ============================================================================

modelo_xgb_ref = joblib.load(RUTA_MODELS / 'XGBoost__none.pkl')
modelo_lgb_ref = joblib.load(RUTA_MODELS / 'LightGBM__none.pkl')

print('Modelos cargados (referencia de configuración):')
print(f'  ✅ XGBoost__none.pkl  → tipo: {type(modelo_xgb_ref).__name__}')
print(f'  ✅ LightGBM__none.pkl → tipo: {type(modelo_lgb_ref).__name__}')


Modelos cargados (referencia de configuración):
  ✅ XGBoost__none.pkl  → tipo: Pipeline
  ✅ LightGBM__none.pkl → tipo: Pipeline


In [3]:
# ============================================================================
# CELDA 3: 5-FOLD CV PAREADO — mismos folds para ambos modelos
# ============================================================================
# Estrategia de CV pareado: usamos StratifiedKFold con random_state=42 (igual
# que F5) para garantizar que XGBoost y LightGBM se entrenan/evalúan EN EL
# MISMO SPLIT. Esto elimina la varianza atribuible a la partición y permite
# comparar las métricas pareadas con el test de Wilcoxon signed-rank.
# ============================================================================

from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone
from sklearn.metrics import roc_auc_score, f1_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

aucs_xgb = []
aucs_lgb = []
f1s_xgb  = []
f1s_lgb  = []

print('5-Fold Stratified CV pareado (random_state=42)')
print('═' * 60)

for fold, (idx_tr, idx_va) in enumerate(cv.split(X_train, y_train), 1):
    X_tr = X_train.iloc[idx_tr]
    X_va = X_train.iloc[idx_va]
    y_tr = y_train.iloc[idx_tr]
    y_va = y_train.iloc[idx_va]
    
    # Reentrenar XGBoost en este fold
    m_xgb = clone(modelo_xgb_ref)
    m_xgb.fit(X_tr, y_tr)
    p_xgb = m_xgb.predict_proba(X_va)[:, 1]
    pred_xgb = (p_xgb >= 0.5).astype(int)
    auc_xgb = roc_auc_score(y_va, p_xgb)
    f1_xgb_val = f1_score(y_va, pred_xgb)
    
    # Reentrenar LightGBM en el MISMO fold
    m_lgb = clone(modelo_lgb_ref)
    m_lgb.fit(X_tr, y_tr)
    p_lgb = m_lgb.predict_proba(X_va)[:, 1]
    pred_lgb = (p_lgb >= 0.5).astype(int)
    auc_lgb = roc_auc_score(y_va, p_lgb)
    f1_lgb_val = f1_score(y_va, pred_lgb)
    
    aucs_xgb.append(auc_xgb); aucs_lgb.append(auc_lgb)
    f1s_xgb.append(f1_xgb_val); f1s_lgb.append(f1_lgb_val)
    
    diff = auc_xgb - auc_lgb
    flecha = '↑' if diff > 0 else '↓' if diff < 0 else '='
    print(f'  Fold {fold}: XGB AUC={auc_xgb:.4f}  LGB AUC={auc_lgb:.4f}  Δ={diff:+.4f} {flecha}')

print('═' * 60)
print(f'\nXGBoost  AUC: {np.mean(aucs_xgb):.4f} ± {np.std(aucs_xgb):.4f}')
print(f'LightGBM AUC: {np.mean(aucs_lgb):.4f} ± {np.std(aucs_lgb):.4f}')
print(f'Diferencia media (XGB - LGB): {np.mean(aucs_xgb) - np.mean(aucs_lgb):+.4f}')


5-Fold Stratified CV pareado (random_state=42)
════════════════════════════════════════════════════════════
  Fold 1: XGB AUC=0.9536  LGB AUC=0.9520  Δ=+0.0016 ↑
  Fold 2: XGB AUC=0.9530  LGB AUC=0.9516  Δ=+0.0014 ↑
  Fold 3: XGB AUC=0.9560  LGB AUC=0.9541  Δ=+0.0019 ↑
  Fold 4: XGB AUC=0.9508  LGB AUC=0.9500  Δ=+0.0007 ↑
  Fold 5: XGB AUC=0.9539  LGB AUC=0.9526  Δ=+0.0012 ↑
════════════════════════════════════════════════════════════

XGBoost  AUC: 0.9535 ± 0.0017
LightGBM AUC: 0.9521 ± 0.0013
Diferencia media (XGB - LGB): +0.0014


In [4]:
# ============================================================================
# CELDA 4: TEST DE WILCOXON SIGNED-RANK PAREADO
# ============================================================================
# Con los 5 valores AUC y F1 pareados (mismo fold), aplicamos el test no
# paramétrico de Wilcoxon signed-rank. Es la versión pareada del test U
# de Mann-Whitney y se usa cuando los datos no son normales o n es pequeño.
# ============================================================================

from scipy import stats

stat_auc, p_auc = stats.wilcoxon(aucs_xgb, aucs_lgb, alternative='two-sided')
stat_f1,  p_f1  = stats.wilcoxon(f1s_xgb,  f1s_lgb,  alternative='two-sided')

print('═' * 60)
print('TEST DE WILCOXON SIGNED-RANK PAREADO (5 folds)')
print('═' * 60)
print()
print(f'  AUC:  W={stat_auc:.4f}  p-value={p_auc:.4f}')
print(f'  F1:   W={stat_f1:.4f}  p-value={p_f1:.4f}')
print()

ALPHA = 0.05
print(f'📊 Veredicto (α = {ALPHA}):')
print()

for nombre, p in [('AUC', p_auc), ('F1', p_f1)]:
    if p < ALPHA:
        veredicto = '❌ DIFERENCIA SIGNIFICATIVA — XGBoost es estadísticamente mejor'
    else:
        veredicto = '✅ EMPATE TÉCNICO — modelos estadísticamente equivalentes'
    print(f'  {nombre}: p={p:.4f}  →  {veredicto}')

print()
empate_total = (p_auc >= ALPHA and p_f1 >= ALPHA)
if empate_total:
    print('🎯 CONCLUSIÓN: Empate técnico en AUC y F1.')
    print('   La elección XGBoost vs LightGBM debe basarse en OTROS criterios:')
    print('   • Estabilidad (auc_std)')
    print('   • Interpretabilidad / fairness')
    print('   • Eficiencia / coherencia con app')
    print('   • Documentación previa en memoria')
else:
    print('🎯 CONCLUSIÓN: Diferencia significativa detectada.')
    print('   Es preferible aceptar XGBoost como ganador.')


════════════════════════════════════════════════════════════
TEST DE WILCOXON SIGNED-RANK PAREADO (5 folds)
════════════════════════════════════════════════════════════

  AUC:  W=0.0000  p-value=0.0625
  F1:   W=1.0000  p-value=0.1250

📊 Veredicto (α = 0.05):

  AUC: p=0.0625  →  ✅ EMPATE TÉCNICO — modelos estadísticamente equivalentes
  F1: p=0.1250  →  ✅ EMPATE TÉCNICO — modelos estadísticamente equivalentes

🎯 CONCLUSIÓN: Empate técnico en AUC y F1.
   La elección XGBoost vs LightGBM debe basarse en OTROS criterios:
   • Estabilidad (auc_std)
   • Interpretabilidad / fairness
   • Eficiencia / coherencia con app
   • Documentación previa en memoria


In [5]:
# ============================================================================
# CELDA 5: GUARDAR RESULTADO PARA REFERENCIAR EN MEMORIA
# ============================================================================

resultado = {
    'fecha':            pd.Timestamp.now().isoformat(),
    'modelos':          ['XGBoost__none', 'LightGBM__none'],
    'cv_folds':         5,
    'random_state':     42,
    'n_train':          int(len(y_train)),
    'aucs_xgb':         [round(x, 6) for x in aucs_xgb],
    'aucs_lgb':         [round(x, 6) for x in aucs_lgb],
    'f1s_xgb':          [round(x, 6) for x in f1s_xgb],
    'f1s_lgb':          [round(x, 6) for x in f1s_lgb],
    'auc_mean_xgb':     float(np.mean(aucs_xgb)),
    'auc_mean_lgb':     float(np.mean(aucs_lgb)),
    'auc_std_xgb':      float(np.std(aucs_xgb)),
    'auc_std_lgb':      float(np.std(aucs_lgb)),
    'wilcoxon_auc':     {'statistic': float(stat_auc), 'p_value': float(p_auc)},
    'wilcoxon_f1':      {'statistic': float(stat_f1),  'p_value': float(p_f1)},
    'alpha':            0.05,
    'conclusion':       'empate' if empate_total else 'xgboost_significativo',
}

ruta_json = RUTA_RESULTS / 'wilcoxon_xgb_vs_lgb.json'
ruta_json.parent.mkdir(parents=True, exist_ok=True)
with open(ruta_json, 'w', encoding='utf-8') as f:
    json.dump(resultado, f, indent=2, ensure_ascii=False)

print(f'💾 Resultado guardado:')
print(f'   {ruta_json}')
print()
print('📝 Cita sugerida para memoria del TFM:')
print()
if empate_total:
    print(f'   "El test de Wilcoxon signed-rank pareado (n=5 folds, α=0.05) entre')
    print(f'    XGBoost y LightGBM no detecta diferencias estadísticamente')
    print(f'    significativas (AUC: p={p_auc:.4f}, F1: p={p_f1:.4f}). Por ello,')
    print(f'    se selecciona LightGBM como modelo final priorizando estabilidad')
    print(f'    CV (auc_std={np.std(aucs_lgb):.4f} vs {np.std(aucs_xgb):.4f}) y coherencia')
    print(f'    con el pipeline F6/aplicación productiva."')
else:
    print(f'   "El test de Wilcoxon signed-rank pareado (n=5 folds, α=0.05)')
    print(f'    detecta una diferencia significativa entre XGBoost y LightGBM')
    print(f'    (AUC: p={p_auc:.4f}, F1: p={p_f1:.4f}). Se selecciona XGBoost')
    print(f'    como modelo final con AUC={np.mean(aucs_xgb):.4f}, F1={np.mean(f1s_xgb):.4f}."')


💾 Resultado guardado:
   c:\PRUEBAS\AU_UJI_v2_RUTA_B\data\05_modelado\results\wilcoxon_xgb_vs_lgb.json

📝 Cita sugerida para memoria del TFM:

   "El test de Wilcoxon signed-rank pareado (n=5 folds, α=0.05) entre
    XGBoost y LightGBM no detecta diferencias estadísticamente
    significativas (AUC: p=0.0625, F1: p=0.1250). Por ello,
    se selecciona LightGBM como modelo final priorizando estabilidad
    CV (auc_std=0.0013 vs 0.0017) y coherencia
    con el pipeline F6/aplicación productiva."
